# WEEK 4 EXERCISE — 3名チームで「解析に必要なデータと背景知識」を備える

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astrocamp-2026-siaPpts/astrocamp-2026-sia/blob/main/exercises/week4_exercise.ipynb)

> **Private リポジトリの注意:** このリポジトリは Private 設定のため、上のバッジから直接開けない場合があります。
> その場合は以下の手順で開いてください：
>
> 1. [Google Colab](https://colab.research.google.com/) にアクセス
> 2. 「ファイル」→「ノートブックを開く」→「GitHub」タブ
> 3. 「プライベート リポジトリを含める」にチェックを入れる
> 4. 初回はGitHub認証が求められるので許可する
> 5. リポジトリ `astrocamp-2026-siaPpts/astrocamp-2026-sia` を選択し、`exercises/week4_exercise.ipynb` を開く
> 6. このノートは保存・提出用ではなく、**チームでの作業メモ帳**として使います。

この演習のゴールは、いきなりコードを書くことではなく、**3名チームで最終課題（相馬市・南相馬市の「作付 / 休耕」判別）を解くために必要なデータと背景知識を備える**ことです。

### 今日のゴール（チームで達成）
1. 3名チーム（camp-alpha / beta / gamma）の役割が決まり、対象エリアの背景と季節サイクルをチームの言葉で説明できる
2. 光学（Sentinel-2）と SAR（Sentinel-1）の最小限の仕様（バンド・偏波・CRS・dB・軌道）を整理できる
3. 公開データ資産（筆ポリゴン・e-Stat）から解析対象を選定できる
4. GEE Code Editorで「エラーなく動くが物理的に不正なAIコード」を「5つの問い」で疑い、正しいデータスタックに直せる
5. 解析計画書（`exercises/analysis_doc.md`）のワークフロー（Step1〜Step3）を、専門家・非専門家に説明できる形で書き始める
6. `camp-[チーム名]` ブランチの使い方を理解し、成果物を `submissions/camp/` に置く

> 💡 **ヒント:** AIで衛星データ解析を行う際の「任せてよいこと vs 検証すべきこと」は、[Zenn「ChatGPTで始める衛星画像処理入門」](https://zenn.dev/syu_tan/books/chatgpt-satellite-image-intro) が参考になります。

---


---
## STEP 0: 3名チームのセットアップ（役割と今日の心構え）

今日の演習は基本的にチームで進めます。まずチーム名と役割を決めてください（チームわけはスライド・README のブランチ構成と対応）。

```
【チームわけ】（チーム名: camp- ＿ ＿ ＿）
- チーム名（camp-alpha / camp-beta / camp-gamma のどれ）: ＿
- 作業ブランチ: ＿（例 camp- ＿）
- 作業ディレクトリ: 例 = submissions/camp/team-alpha/

【3名の役割分担】
- データ調査リーダー（Sentinel-2 / Sentinel-1 / 筆ポリゴン / e-Stat の仕様と取得先を調べる）
  → 名前: ＿
- 方法・背景リーダー（対象エリアの背景・季節サイクル・物理的根拠をまとめる）
  → 名前: ＿
- 発表・ファクトチェック担当（話し合いを整理し、「物理的に正しいか」を最後に疑う）
  → 名前: ＿

【今日のゴール確認】（終了時に ☑ を付ける）
□ 背景知識（PART A）をチームの言葉で説明できた
□ データ知識（PART B ①②③）の仕様表を埋めた
□ 罠を「5つの問い」で狩り、修正した（PART C）
□ 罠の説明を専門家 / 非専門家にできる（共通項＋違い）
□ 解析計画書（PART D / analysis_doc.md）の Step1〜Step3 を書き始めた
□ 成果物を `camp-[チーム名]` ブランチの `submissions/camp/` に置いた
```

> このノートはチームの作業メモ帳として使い、個人の提出物（Exercise Log）は最後のセクションでまとめます。

---


---
## PART A 背景知識① — 対象エリアと「作付 / 休耕」の現象

### 対象エリア
最終課題は、**福島県浜通り**の**相馬市・南相馬市**の農地です。

- 沿岸部の平坦な農地で、水田・畑の作付と休耕（遊休農地）が混在
- 2011年の東日本大震災と東京電力福島第一原子力発電所事故の避難対象地域を含む
- 現在は農地整備（大区画化・土壌の除染）が進み、作付の再開と休耕が併存

> **「作付 / 休耕」判別の「現象」** —— 同じ場所でも季節で姿が変わります。この「変化」を、光学とSARのそれぞれの物理でどう読むかが要点。

### 季節サイクル（水田の例）
| 時期 | 農地の様子 | 光学（NDVI） | SAR（σ⁰・VH） |
| -- | -- | -- | -- |
| 4月 | 代かき前・休耕中 | 低い（裸土） | 中（土の表面散乱） |
| 5月 | 湛水期（水張り） | 低い（水面） | **暗い（鏡面反射）** |
| 6〜7月 | 生育期（葉が育つ） | 上昇 | 高（葉の構造散乱） |
| 8月 | 最盛期・収穫前 | ピーク付近 | 高（体積散乱） |
| 収穫後 | 裸土・稲わら | 低下 | 低下 |

### チーム課題 A
- この現象をチームの言葉で1行に書いてください（「＿＿が＿になるから、作付と休耕が区別できる」）。
- 作付 / 休耕の判別に使う観測の時期と、その理由を1つ挙げてください。


## PART B データ知識① — 光学（Sentinel-2）

### 主要バンド（Sentinel-2）
| バンド | 中心波長 | 種類と用途 |
| -- | -- | -- |
| B2 (Blue) | ~490nm | 大気・水 |
| B3 (Green) | ~560nm | 植被（目視） |
| B4 (Red) | ~665nm | 葉緑素の吸収 → **NDVI の Red** |
| B5 (Red Edge) | ~705nm | 植生の遷移帯（**NDVI の NIR としては使わない**） |
| B8 (NIR) | ~842nm | **NDVI の NIR**・葉の構造反射 |
| B8A (Narrow NIR) | ~865nm | 狭帯域 NIR（用途による） |
| B11 / B12 (SWIR) | ~1610 / 2190nm | 水分量・土壌 |

**NDVI = (NIR − Red) ÷ (NIR + Red) = (B8 − B4) ÷ (B8 + B4)**

### 押さえる 3 点
- **正しいバンド**：NDVI は **B8（NIR）と B4（Red）**。B5（Red Edge）を NIR にすると NDVI が過小評価され、作付中の農地まで「休耕」に見えます。
- **CRS（座標系）**：Sentinel-2 は UTM（EPSG:32654 など）。外部ポリゴンと揃えないと「ピクセルと境界」がズレます。
- **雲量フィルタ**：曇りシーンを除外する（`CLOUDY_PIXEL_PERCENTAGE < 20`）。

### チーム課題 B1
- NDVI の式（B8 / B4）を自分の手で書く。
- 「B5 を NIR に使うとどうなるか」「CRS を揃えないとどうなるか」を一言ずつ。


## PART B データ知識② — SAR（Sentinel-1）

### 重要なポイント
- **偏波**: Sentinel-1 GRD は **VV と VH**（デュアル偏波）。HH は Sentinel-1 では使えません。
- **観測モード**: 農業解析には **IW**（解像度 10m）が最適。
- **単位**: `COPERNICUS/S1_GRD` は**すでに dB（σ⁰）**で格納。`10*log10()` をさらに適用すると「二重変換」で値域が壊れます。
- **軌道方向**: 昇交（ASCENDING）と降交（DESCENDING）を混ぜると、時系列の見かけが揺れます。**どちらかに統一**します。
- **スペックル**: 粒状のノイズ。圃場内の平均・中央値（median）で低減します。

### 地面の見え方（散乱機構）
| 状況 | 見え方（σ⁰） |
| -- | -- |
| 水面（湛水期） | **非常に暗い**（鏡面反射） |
| 湿った土 | 暗い |
| 背の高い作物 | **明るい**（体積散乱） |
| 建物・ダブルバウンス | とても明るい |

### チーム課題 B2（各一言）
- VV と VH の違い／`S1_GRD` が dB なのに `10*log10()` をしてはいけない理由／昇交・降交を揃えないと時系列で何が起こるか／スペックルへの対応例。


## PART B データ知識③ — 解析を支える公開データ

- **筆ポリゴン**（農林水産省、`open.fude.maff.go.jp`）: 1筆ごとの境界ポリゴン。圃場単位の集計に最適。座標系に注意（JGD2011 系 等）。
- **e-Stat**（[https://www.e-stat.go.jp/](https://www.e-stat.go.jp/)）: 農業集落・耕地面積の統計。対象エリアの選定に使います。

### 座標系の整理
| 座標系 | 内容 |
| -- | -- |
| WGS84（地理座標） | 経緯度。bbox は [経度, 緯度] |
| UTM (EPSG:32654) | 日本の一般的な UTM。Sentinel-2 など |
| JGD2011 | 日本の国家座標（筆ポリゴンなど） |

### チーム課題 B3（e-Stat で候補エリアを調べる）
- 相馬市・南相馬市から候補エリアを 1〜2 箇所選び、選んだ理由を 2〜3 行で書きましょう。


## PART C 手を動かす① — GEE Code Editor で「AI生成コード（罠入り）」を実行する（約12分）

配布 `exercises/week4_gee.js` を [GEE Code Editor](https://code.earthengine.google.com/) にコピー＆ペーストして **Run** してください。**このコードはエラーが出ません**。しかし **3つの罠**（物理的に不正な箇所）が埋め込まれています。「エラーが出ない＝正しい」ではありません。

```javascript
// 福島県浜通り（相馬市・南相馬市周辺の農地）
var roi = ee.Geometry.Rectangle([140.85, 37.55, 141.02, 37.80]);

// --- 1. 光学（Sentinel-2）：NDVI 最大値抽出 ---
var s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(roi).filterDate('2024-06-01', '2024-08-31')
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20));
// 【罠①】B5（Red Edge）を NIR と誤用 → NDVI 値の過小

var ndvi_max = s2.map(function(img) {
  return img.normalizedDifference(['B5', 'B4']).rename('NDVI');
}).max().clip(roi);

// --- 2. SAR（Sentinel-1）：時系列変化量 ---
var s1 = ee.ImageCollection('COPERNICUS/S1_GRD')
  .filterBounds(roi)
  .filterDate('2024-08-01', '2024-08-31')
  .filter(ee.Filter.eq('instrumentMode', 'IW'));
// 【罠②】昇交/降交（ASCENDING / DESCENDING）を統一していない

var s1_vh_mean = s1.select('VH').mean().clip(roi);

// --- 3. （疑似ラベルによる）休耕地判定 ---
// 【罠③】S1_GRD はすでに dB なのに 10*log10() をさらに実行し、かつ閾値 -20 が物理的に破綻
var fallow_mask = ndvi_max.lt(0.35).and(s1_vh_mean.log10().multiply(10).lt(-20));

// --- 4. 可視化 ---
Map.centerObject(roi, 12);
Map.addLayer(ndvi_max, {min: 0, max: 0.8, palette: ['blue','white','green']}, 'NDVI Max (罠①)');
Map.addLayer(s1_vh_mean, {min: -25, max: -5}, 'SAR VH Mean (罠②)');
Map.addLayer(fallow_mask.updateMask(fallow_mask), {palette: ['red']}, '休耕地疑い (罠③)');
```

**実行の手順**
1. コピー＆ペースト → **Run**（エラーが出ないことを確認）
2. 右の **Inspector** で農地（大区画）をクリックし、NDVI・VH の値を計測
3. 「PART A の季節サイクル」の表と照らし合わせ、値の違和感を探す


## PART C 手を動かす② — 正しいデータスタックに修正する（約20分）

| 罠 | 不正な実装 | 正しい実装 | 修正理由（物理的根拠） |
| -- | -- | -- | -- |
| ① NDVI | `['B5','B4']` | `['B8','B4']` | NDVI の NIR は B8。B5（RedEdge）は遷移帯で構造散乱を捉えず NDVI 過小 |
| ② 軌道 | 昇交/降交を混在 | `filter(ee.Filter.eq('orbitProperties_pass','DESCENDING'))` などで統一 | 入射角が揃わないと時系列 σ⁰ が揺れるため。湛水と乾燥を誤読 |
| ③ 二重変換 | `log10(...)*10 < -20` | dB のまま閾値（median でスペックル低減後） | S1_GRD は既に dB。二重変換で値域が壊れるため |

**修正した正しいコード（ヒント）**
```javascript
var s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(roi).filterDate('2024-06-01','2024-08-31')
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20));
var ndvi_max = s2.map(function(im) {
  return im.normalizedDifference(['B8','B4']).rename('NDVI'); // 罠①修正
}).max().clip(roi);

var s1 = ee.ImageCollection('COPERNICUS/S1_GRD')
  .filterBounds(roi).filterDate('2024-08-01','2024-08-31')
  .filter(ee.Filter.eq('instrumentMode','IW'))
  .filter(ee.Filter.eq('orbitProperties_pass','DESCENDING')); // 罠②修正
var s1_vh_med = s1.select('VH').median().clip(roi); // median でスペックル低減

var fallow_mask = ndvi_max.lt(0.35).and(s1_vh_med.lt(-18)); // 罠③: dB のまま閾値
```

### チーム課題 C2
- 各罠を、修正前後で「どう変わるか」「放置するとどの誤判定になるか」を物理的根拠付きで説明してください。


## PART C 手を動かす③ — AIへの「5つの問い」で批判的に確認する

生成AIの出力を「そのまま信じる」のではなく、必ず以下の **5 つの問い** を立てて疑います。エラーがなくても、これらを**コードの該当行と数値**で確認するのがファクトチェックです。

1. **単位は？** … `S1_GRD` はすでに dB なのに `10*log10()` をさらに足していないか
2. **バンド・偏波は？** … NDVI は `B8` & `B4`？ SAR は `VV` / `VH`？ HH を指定していないか
3. **座標系（CRS）は？** … 画像と外部ポリゴンの CRS（EPSG:32654 / JGD2011 等）は揃っているか？
4. **軌道・観測モードは？** … ASC / DESC を統一しているか？観測モードは IW？
5. **観測時期・雲量は？** … 季節サイクルに合う時期か？ 雲量フィルタはあるか？

> **確認方法**: 該当行を見つける → GEE の **Inspector** で数値を計測 → 物理的根拠（散乱機構・スペクトル）と照合。納得するまで疑います。

### チーム課題 C3（専門家 / 非専門家に伝える：共通項＋違い）
罠①〜③のそれぞれについて、**5つの問いのどれで気づいたか**と、**専門家向け・非専門家向け・共通**の説明をまとめます。
```
【罠それぞれについて】
- 5つのうちどれで見つけたか → ＿
- 専門家向け（物理的理由・不確実性）→ ＿
- 非専門家向け（現場の誰の何が誤る）→ ＿
- どちらにも共通すること（目的・根拠・簡単な説明）→ ＿
```


## PART D — 解析計画書（`exercises/analysis_doc.md`）を書く

今日の「正しいデータスタック」は解析計画書の **Step 3（データ・前処理）** の土台です。解析計画書は **5 ステップ** で組み立てます（記入テンプレートは `exercises/analysis_doc.md`）。

| Step | 内容 | 観点 |
| -- | -- | -- |
| Step1 課題 | 誰の・何の課題を解決するか | 非専門家 |
| Step2 物理仮説 | なぜ光学＋SAR で判別できるか | 専門家 |
| Step3 データ | 使用データ・前処理（CRS・軌道・dB） | 専門家 |
| Step4 手法 | 時系列 / 変化検出 / 特徴量スタック / ML | 専門家 |
| Step5 検証・社会実装 | 誤りをどう検出するか（混同行列など）＋費用対効果・現場コスト削減 | 両方 |

### チーム課題 D（チームで記入）
```
【Step1 課題】誰の・何の課題を解決するか → ＿
【Step2 仮説】なぜ光学＋SAR を組み合わせるか（湛水×生長・雲に強い） → ＿
【Step3 データ】Sentinel-2 B8/B4・Sentinel-1 VV/VH・軌道統一・reproject… → ＿
【Step4 手法】時系列 → 変化検出 → 特徴量スタック → ML から絞り込む → ＿
【Step5 検証・社会実装】誤りをどう検出するか（混同行列など）＋費用対効果・現場コスト削減の可能性 → ＿
【専門家 / 非専門家に共通の点】目的・根拠・限界をどう伝えるか → ＿
```
**提出する場所は `camp-[チーム名]` ブランチの `submissions/camp/` です。**


## 提出方法と Exercise Log

このワークシートはチームで記入して進める**チームの成果物**です。**個人の Exercise Log** とは分けて扱います。

- チームの成果物（解析計画書 `analysis_doc.md`・修正したコード）: **`camp-[チーム名]` ブランチ / `submissions/camp/team-alpha/` 等**
- 個人の Exercise Log: `submissions/individual/<あなたのID>/week4.md`（様式は `exercises/template.md`）

**ブランチの使い方（おさらい）**
- 合宿プロジェクトは **自分のチームの `camp-[チーム名]` ブランチ** で、成果物を `submissions/camp/` に置いてプッシュする。
- 個人提出は `submissions`（`submissions/individual/<ID>/`）、事前学習のチームは `prelim-a` / `prelim-b`。
- ブランチ切替・プッシュ手順は `WORKFLOW.md` / `resources/github-guide.md`。

```
【Exercise Log: Week 4 に書くこと】
① 「データと背景知識」の整理結果（季節サイクル・センサ仕様・候補エリア）
② スカベンジャーで見つけた罠を「5つの問い」で検証した結果（専門家視点）と誤判断の例（非専門家視点）
③ 専門家 / 非専門家の説明（共通項＋違い）に関する気づき
④ 解析計画書（analysis_doc）の Step1〜Step3 の記入結果
```

**おつかれさまでした。** 今回、作付 / 休耕の判別に必要なデータ・背景知識と、解析計画書（Step1〜Step3）の書き始めを行いました。次回はこの解析計画書を完成させます。
